<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [7]:
# use requests.get() method with the provided static_url and headers
import requests

# 1. Define the URL
url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"

# 2. Add a User-Agent header to pretend you are a web browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 3. Perform the HTTP GET request with the headers included
response = requests.get(url, headers=headers)

# 4. Check if the request was successful
if response.status_code == 200:
    print("Success! The page was retrieved.")
    # Print the first 500 characters of the page's HTML
    print(response.text[:500])
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")

Success! The page was retrieved.
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp


Create a `BeautifulSoup` object from the HTML `response`


In [8]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
from bs4 import BeautifulSoup

# 1. Verify you have the response text from the previous step
# (Assuming 'response' is the variable holding your successful GET request)
html_content = response.text

# 2. Create the BeautifulSoup object and specify the parser
soup = BeautifulSoup(html_content, 'html.parser')

# 3. Verify it worked by printing out the page title
print("Page Title:", soup.title.string)

Page Title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


Print the page title to verify if the `BeautifulSoup` object was created properly 


In [9]:
# Print the page title string
print(soup.title.string)

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [11]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
# Find all tables with the class 'wikitable'
html_tables = soup.find_all('table', class_='wikitable')

# Print the total number of tables found
print(f"Total wikitables found on the page: {len(html_tables)}")
# Target the first launch table
first_table = html_tables[0]

# Find all header elements (<th>) inside this table
headers = first_table.find_all('th')

# Extract and clean the text from each header
for i, header in enumerate(headers):
    print(f"Index {i}: {header.text.strip()}")

Total wikitables found on the page: 4
Index 0: Flight No.
Index 1: Date andtime (UTC)
Index 2: Version,booster[j]
Index 3: Launchsite
Index 4: Payload[k]
Index 5: Payload mass
Index 6: Orbit
Index 7: Customer
Index 8: Launchoutcome
Index 9: Boosterlanding
Index 10: 418
Index 11: 419
Index 12: 420
Index 13: 421
Index 14: 422
Index 15: 423
Index 16: 424
Index 17: 425
Index 18: 426
Index 19: 427
Index 20: 428
Index 21: 429
Index 22: 430
Index 23: 431
Index 24: 432
Index 25: 433
Index 26: 434
Index 27: 435
Index 28: 436
Index 29: 437
Index 30: 438
Index 31: 439
Index 32: 440
Index 33: 441
Index 34: 442
Index 35: 443
Index 36: 444
Index 37: 445
Index 38: 446
Index 39: 447
Index 40: 448
Index 41: 449
Index 42: 450
Index 43: 451
Index 44: 452
Index 45: 453
Index 46: 454
Index 47: 455
Index 48: 456
Index 49: 457
Index 50: 458
Index 51: 459
Index 52: 460
Index 53: 461
Index 54: 462
Index 55: 463
Index 56: 464
Index 57: 465
Index 58: 466
Index 59: 467
Index 60: 468
Index 61: 469
Index 62: 470
In

Starting from the third table is our target table contains the actual launch records.


In [13]:
# Let's print the third table and check its content
# Target the third table (index 2)
target_table = html_tables[2]

# Find all header elements inside this table
headers = target_table.find_all('th')

# Print raw headers with their index
column_names = []

# Iterate through each header element
for header in headers:
    name = header.text.strip()
    
    # Filter conditions: 
    # 1. Ensure the name is not empty
    # 2. Exclude headers that are just reference numbers or row indices
    if name and len(name) > 0 and not name.isdigit():
        column_names.append(name)

# Print your clean list of variables/columns
print("\nCleaned Column Names:")
print(column_names)


Cleaned Column Names:
['Date and time (UTC)', 'Version,booster[j]', 'Launch site', 'Payload[k]', 'Orbit', 'Customer']


You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [14]:
column_names = []


# Target the 3rd table on the page (index 2)
target_table = html_tables[2]

# Find all <th> elements in this table
th_elements = target_table.find_all('th')

# Iterate through each <th> element one by one
for th in th_elements:
    name = extract_column_from_header(th)
    
    # Append valid, non-empty names to our list
    if name is not None and len(name) > 0:
        column_names.append(name)

# Print the final verified variables
print("Successfully Extracted Column Names:")

# Apply find_all() function with `th` element on first_launch_table
# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name (`if name is not None and len(name) > 0`) into a list called column_names


Successfully Extracted Column Names:


Check the extracted column names


In [15]:
print(column_names)

['Date and time ( )', 'Launch site', 'Payload', 'Orbit', 'Customer']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [19]:
# 1. Initialize an empty dictionary
launch_dict = {}

# 2. Automatically populate keys from your extracted column_names
for name in column_names:
    launch_dict[name] = []

# 3. Add or normalize structural keys to make data cleaning easier later
# Target lists often require explicit sub-components separated:
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []

# Added columns to split Date and Time for better analysis
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

# 4. Let's verify the keys in our dictionary
print("Initialized dictionary keys:")
print(list(launch_dict.keys()))

Initialized dictionary keys:
['Date and time ( )', 'Launch site', 'Payload', 'Orbit', 'Customer', 'Flight No.', 'Payload mass', 'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time']


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [22]:
extracted_row = 0

# Extract each table 
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    # get table row 
    for rows in table.find_all("tr"):
        # check to see if first table heading is a number corresponding to a launch number 
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
            
        # get table elements 
        row = rows.find_all('td')
        
        # if it is a valid launch number, save cells in the dictionary 
        if flag:
            extracted_row += 1
            
            # --- Flight Number ---
            # TODO: Append the flight_number into launch_dict with key `Flight No.`
            launch_dict['Flight No.'].append(flight_number)
            
            # Extract date and time components using the template's helper function
            datatimelist = date_time(row[0])
            
            # --- Date value ---
            # TODO: Append the date into launch_dict with key `Date`
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            
            # --- Time value ---
            # TODO: Append the time into launch_dict with key `Time`
            time = datatimelist[1]
            launch_dict['Time'].append(time)
              
            # --- Booster version ---
            # TODO: Append the bv into launch_dict with key `Version Booster`
            bv = booster_version(row[1])
            if not(bv):
                bv = row[1].a.string if row[1].a else row[1].text.strip()
            launch_dict['Version Booster'].append(bv)
            
            # --- Launch Site ---
            # TODO: Append the launch_site into launch_dict with key `Launch site`
            launch_site = row[2].a.string if row[2].a else row[2].text.strip()
            launch_dict['Launch site'].append(launch_site)
            
            # --- Payload ---
            # TODO: Append the payload into launch_dict with key `Payload`
            payload = row[3].a.string if row[3].a else row[3].text.strip()
            launch_dict['Payload'].append(payload)
            
            # --- Payload Mass ---
            # TODO: Append the payload_mass into launch_dict with key `Payload mass`
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # --- Orbit ---
            # TODO: Append the orbit into launch_dict with key `Orbit`
            orbit = row[5].a.string if row[5].a else row[5].text.strip()
            launch_dict['Orbit'].append(orbit)
            
            # --- Customer ---
            # TODO: Append the customer into launch_dict with key `Customer`
            # Safety Check: Handled cases where there is no hyperlink (<a>) tag present
            if row[6].a:
                customer = row[6].a.string
            else:
                customer = row[6].text.strip()
            launch_dict['Customer'].append(customer)
            
            # --- Launch outcome ---
            # TODO: Append the launch_outcome into launch_dict with key `Launch outcome`
            launch_outcome = list(row[7].strings)[0].strip() if row[7].strings else row[7].text.strip()
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # --- Booster landing ---
            # TODO: Append the booster_landing into launch_dict with key `Booster landing`
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Data mapping complete! Successfully parsed {extracted_row} flight records.")

Data mapping complete! Successfully parsed 0 flight records.


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [24]:
import pandas as pd

# 1. Convert the launch_dict into a Pandas DataFrame safely
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})

# 2. View the dimensions of your newly created dataset
print(f"DataFrame successfully created! Shape: {df.shape} (Rows, Columns)\n")

# 3. Display the first 5 records to inspect the structural alignment
print("--- FIRST 5 LAUNCH RECORDS ---")
print(df.head(5))


DataFrame successfully created! Shape: (0, 12) (Rows, Columns)

--- FIRST 5 LAUNCH RECORDS ---
Empty DataFrame
Columns: [Date and time ( ), Launch site, Payload, Orbit, Customer, Flight No., Payload mass, Launch outcome, Version Booster, Booster landing, Date, Time]
Index: []


/home/jupyterlab/conda/envs/python/lib/python3.7/site-packages/ipykernel_launcher.py:4: DeprecationWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  after removing the cwd from sys.path.


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
